# 02 · Preprocessing
Walks through the preprocessing pipeline implemented in `src/data_prep.py`.

**Design principle:** every data-dependent step (imputation, scaling, encoding) lives *inside* a scikit-learn `Pipeline`, so it is re-fitted on each training fold and no information leaks from validation/test data.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.data_prep import *
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

In [2]:
df = clean(load_raw())
X, y = df[FEATURES], df[TARGET]
print('Feature groups:'); print(' numeric    :', NUMERIC); print(' categorical:', CATEGORICAL); print(' binary     :', BINARY)

Feature groups:
 numeric    : ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
 categorical: ['cp', 'restecg', 'slope', 'thal']
 binary     : ['sex', 'fbs', 'exang']


## Train / test split
Stratified 80/20 split. The test set is locked away until final evaluation; all tuning uses cross-validation on the training portion.

In [3]:
X_train, X_test, y_train, y_test = get_splits()
print(X_train.shape, X_test.shape)
print('Train prevalence:', y_train.mean().round(3), '| Test prevalence:', y_test.mean().round(3))

(242, 13) (61, 13)
Train prevalence: 0.459 | Test prevalence: 0.459


## Preprocessor
- Numerics (incl. `ca`): median imputation → standard scaling
- Categoricals (`cp`, `restecg`, `slope`, `thal`): mode imputation → one-hot
- Binary: passed through

In [4]:
prep = build_preprocessor()
Xt = prep.fit_transform(X_train)
print('Transformed shape:', Xt.shape)
display(Xt.head())
print('Any NaN after transform?', Xt.isna().any().any())

Transformed shape: (242, 22)


,age,trestbps,chol,thalach,oldpeak,ca,cp_1.0,cp_2.0,cp_3.0,cp_4.0,restecg_0.0,restecg_1.0,restecg_2.0,slope_1.0,slope_2.0,slope_3.0,thal_3.0,thal_6.0,thal_7.0,sex,fbs,exang
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,-0.689715,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1,0,0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,-0.689715,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1,0,0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,0.445734,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0,1,1
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,-0.689715,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1,0,0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,-0.689715,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0,0,1


Any NaN after transform? False


In [5]:
display(Xt[NUMERIC].describe().loc[['mean','std']].round(2))

,age,trestbps,chol,thalach,oldpeak,ca
mean,-0.0,-0.0,0.0,-0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0


Numeric features now have mean ≈ 0 and std ≈ 1 (fitted on the training set only). Save the cleaned dataset with `python -m src.data_prep`.